In [0]:
%pip install soda-core-spark-df

In [0]:
# =============================================================================
# F1-Pulse | Data Quality — Gold Layer
# Notebook: 05_Gold_Quality
# Author:   Jafar891
# Updated:  2026
#
# Runs Soda checks against Gold Delta tables.
# Raises RuntimeError on failure to halt the Databricks job.
# =============================================================================

import logging

from soda.scan import Scan

from config.config import CATALOG, GOLD

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
log = logging.getLogger("f1_pulse.dq.gold")

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
REPO_ROOT   = "/Workspace/Users/jafar.gahramanov@gmail.com/F1-Pulse"
CHECKS_FILE = f"{REPO_ROOT}/soda/checks_gold.yml"

# ---------------------------------------------------------------------------
# Load Gold tables and register as temp views
# Soda requires temp views — dots in table names are not allowed in view names
# ---------------------------------------------------------------------------
log.info("Registering Gold temp views …")
spark.table(f"{CATALOG}.{GOLD}.driver_performance_metrics").createOrReplaceTempView("gold_driver_performance_metrics")
spark.table(f"{CATALOG}.{GOLD}.constructor_standings").createOrReplaceTempView("gold_constructor_standings")
spark.table(f"{CATALOG}.{GOLD}.lap_progression").createOrReplaceTempView("gold_lap_progression")

# ---------------------------------------------------------------------------
# Load and patch checks YAML
# Patches applied:
#   - Table names → temp view names (dots replaced with underscores)
# ---------------------------------------------------------------------------
log.info(f"Loading checks from {CHECKS_FILE} …")
with open(CHECKS_FILE, "r") as f:
    yaml_content = f.read()

yaml_content = yaml_content.replace(
    "checks for gold.driver_performance_metrics",
    "checks for gold_driver_performance_metrics",
)
yaml_content = yaml_content.replace(
    "checks for gold.constructor_standings",
    "checks for gold_constructor_standings",
)
yaml_content = yaml_content.replace(
    "checks for gold.lap_progression",
    "checks for gold_lap_progression",
)

# ---------------------------------------------------------------------------
# Run Soda scan
# ---------------------------------------------------------------------------
log.info("Running Soda scan …")
scan = Scan()
scan.set_data_source_name("spark_df")
scan.add_spark_session(spark)
scan.add_sodacl_yaml_str(yaml_content)

exit_code = scan.execute()
log.info(scan.get_logs_text())

# ---------------------------------------------------------------------------
# Halt pipeline on failure — Databricks job marks task as FAILED
# ---------------------------------------------------------------------------
if exit_code != 0:
    raise RuntimeError("Gold DQ checks failed — halting pipeline.")

log.info("✅ Gold DQ checks passed.")